## 📓 About this notebook
This notebook stress-tests the Unity Airways agent with simulated multi-turn conversations and scores them through one unified MLflow evaluation loop that mixes built-in scorers with a custom LLM judge.

**Maps to the book:** Chapter 10, *Unifying GenAI Systems with MLflow* — sections: Integrating the Ecosystem Through Third-Party Judges, Road Ahead.

In [ ]:
%pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

## Conversation Simulation

### Define simulated scenarios
When real conversation logs are scarce, `ConversationSimulator` generates multi-turn dialogues from goal + persona test cases, driven by a user model, giving a systematic way to evaluate agent behavior. _(see Ch 10, "Road Ahead")_

### Define the Scenarios

### Add a custom judge
`make_judge` defines a bespoke LLM judge (here, professional tone) that plugs into the same scorer interface as native and third-party judges, keeping evaluation in one shared loop. _(see Ch 10, "Integrating the Ecosystem Through Third-Party Judges")_

In [0]:
import mlflow
from mlflow.genai.simulators import ConversationSimulator
from mlflow.genai.scorers import ConversationCompleteness, Safety

# Unity Air Test Cases
test_cases = [
    {
        "goal": "Reschedule booking id xaji0y6dpbhsahxt, confirm the fare difference and whether the LITE fare brand allows changes, and handle the expected rejection gracefully",
        "persona": "A frequent business traveler who is direct and impatient, expects quick answers, and pushes back if the agent is vague about costs or policies.",                                                                                           
    },
    {
      "goal": "Ask about the refund status for booking id xxc02dgtfgq5c34d. Verify that the refund has been processed",
      "persona": "A non-technical traveler who speaks in casual language, sometimes provides incomplete information, and needs the agent to ask clarifying questions."
    },
]

# Conversation Simulator
simulator = ConversationSimulator(
    test_cases=test_cases,
    max_turns=2,
    user_model = "databricks:/databricks-gpt-oss-120b"
)

### Load the registered agent
Resolve and load the latest version of the tool-calling agent from Unity Catalog so the simulator can run conversations against the real model. _(see Ch 10, "Integrating the Ecosystem Through Third-Party Judges")_

### Custom Scorer

In [0]:
from mlflow.genai.judges import make_judge                                                                                    
from typing import Literal                                                                                                    
                                                                                                                                
professional_tone_judge = make_judge(
    name="professional_tone",                                                                                                 
    instructions=(
        "Review this conversation:\n\n{{ conversation }}\n\n"
        "Evaluate whether the assistant maintained a professional, courteous, "
        "and airline-appropriate tone throughout the conversation. "
        "Consider: empathy when handling complaints, clear and polite language, "
        "and avoiding overly casual or robotic responses.\n\n"
        "Rate as 'professional' or 'unprofessional'."
    ),
    feedback_value_type=bool,
)

## Load Agent
Let's load the agent from Chapter 4 and see it performs

### Run the unified evaluation loop
`mlflow.genai.evaluate` runs the simulated conversations through built-in scorers and the custom judge together, keeping native and external evaluation on one shared results surface. _(see Ch 10, "Integrating the Ecosystem Through Third-Party Judges")_

In [0]:
import yaml 

uc_model_conf_path = "../conf/uc_model_registry.yml"
with open(uc_model_conf_path, "r") as f:
    uc_model_conf = yaml.safe_load(f)

In [0]:
from mlflow.tracking.client import MlflowClient

def get_latest_model_version(model_name):
  client = MlflowClient()
  model_version_infos = client.search_model_versions("name = '%s'" % model_name)
  return max([int(model_version_info.version) for model_version_info in model_version_infos])

In [0]:
tool_calling_agent_uc = uc_model_conf.get('tool_calling_agent').get('full_name')
latest_version = get_latest_model_version(tool_calling_agent_uc)
agent = mlflow.langchain.load_model(f"models:/{tool_calling_agent_uc}/{latest_version}")

In [0]:
def predict_fn(input: list[dict], **kwargs) -> str:
    output = agent.invoke({"messages": input})
    msgs = output['messages']
    last_msg = msgs[-1].model_dump()
    return last_msg

In [0]:
# Simulate conversations and evaluate
results = mlflow.genai.evaluate(
    data=simulator,
    predict_fn=predict_fn,
    scorers=[
        ConversationCompleteness(),
        Safety(),
        professional_tone_judge
    ],
)